In [1]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Function

# ==========================================
# 1. 환경 및 디바이스 설정
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"INFO: 현재 사용 중인 디바이스: {device}")

# 구글 코랩 사용 시 드라이브 마운트 (로컬 Jupyter 환경이면 주석 처리)
from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# 2. 핵심 아키텍처 정의 (베이스라인, Fusion, DANN)
# ==========================================
def build_resnet50_model(num_classes: int, pretrained: bool = False) -> nn.Module:
    weights = models.ResNet50_Weights.DEFAULT if pretrained else None
    model = models.resnet50(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

class AgriAX_FusionModel(nn.Module):
    def __init__(self, drone_model: nn.Module, satellite_input_dim: int, num_classes: int):
        super(AgriAX_FusionModel, self).__init__()
        self.drone_features = nn.Sequential(*list(drone_model.children())[:-1])
        self.drone_fc_dim = drone_model.fc.in_features
        self.satellite_encoder = nn.Sequential(
            nn.Linear(satellite_input_dim, 64), nn.ReLU(), nn.Dropout(0.3)
        )
        self.fusion_layer = nn.Sequential(
            nn.Linear(self.drone_fc_dim + 64, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, num_classes)
        )

    def forward(self, drone_img, satellite_data):
        d_feat = torch.flatten(self.drone_features(drone_img), 1)
        s_feat = self.satellite_encoder(satellite_data)
        return self.fusion_layer(torch.cat((d_feat, s_feat), dim=1))

class GradientReversalLayer(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class AgriAX_DANN(nn.Module):
    def __init__(self, base_model, num_classes):
        super(AgriAX_DANN, self).__init__()
        self.feature_extractor = nn.Sequential(*list(base_model.children())[:-1])
        self.class_classifier = nn.Linear(base_model.fc.in_features, num_classes)
        self.domain_classifier = nn.Sequential(
            nn.Linear(base_model.fc.in_features, 256), nn.ReLU(), nn.Linear(256, 2)
        )

    def forward(self, x, alpha=1.0):
        features = torch.flatten(self.feature_extractor(x), 1)
        class_output = self.class_classifier(features)
        domain_output = self.domain_classifier(GradientReversalLayer.apply(features, alpha))
        return class_output, domain_output

# ==========================================
# 3. 모델 가중치 로드 및 초기화
# ==========================================
NUM_CLASSES = 38 # 기존 PlantVillage 클래스 수

# 코랩 환경 예시:
WEIGHT_PATH = '/content/drive/Othercomputers/내 노트북/AgriAX/models/baseline_resnet50.pth'
# 로컬(C드라이브) 환경 예시:
# WEIGHT_PATH = 'C:/DL_DATA/models/baseline_resnet50.pth'

model = build_resnet50_model(num_classes=NUM_CLASSES)

try:
    if os.path.exists(WEIGHT_PATH):
        model.load_state_dict(torch.load(WEIGHT_PATH, map_location=device))
        model.to(device)
        model.eval()
        print("INFO: Day 1-2 베이스라인 모델(ResNet50) 가중치 복구 완료.")
    else:
        print(f"WARNING: 가중치 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요: {WEIGHT_PATH}")
except Exception as e:
    print(f"ERROR: 가중치 로드 중 오류 발생: {e}")

# ==========================================
# 4. 공통 이미지 전처리 (Transform)
# ==========================================
target_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print("INFO: 공통 초기 세팅이 모두 완료되었습니다. 다음 작업을 진행하세요.")

INFO: 현재 사용 중인 디바이스: cuda
Mounted at /content/drive
INFO: Day 1-2 베이스라인 모델(ResNet50) 가중치 복구 완료.
INFO: 공통 초기 세팅이 모두 완료되었습니다. 다음 작업을 진행하세요.


In [4]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

# 업로드한 미니 데이터셋 경로 지정
TARGET_DATA_DIR = '/content/drive/MyDrive/mini_peppers_matched'

target_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class AIHubMatchedDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_dir = os.path.join(root_dir, 'val', 'image')
        self.label_dir = os.path.join(root_dir, 'val', 'labels')
        self.transform = transform
        self.valid_data = []

        if not os.path.exists(self.image_dir):
            raise FileNotFoundError(f"경로를 찾을 수 없습니다: {self.image_dir}")

        images = [f for f in os.listdir(self.image_dir) if f.endswith(('.jpg', '.jpeg', '.png', '.JPG'))]

        for img_name in images:
            # 추출 스크립트에서 사용한 두 가지 라벨 명명 규칙 대응
            label_name_1 = img_name + '.json'
            label_name_2 = os.path.splitext(img_name)[0] + '.json'

            lbl_path_1 = os.path.join(self.label_dir, label_name_1)
            lbl_path_2 = os.path.join(self.label_dir, label_name_2)

            lbl_path = lbl_path_1 if os.path.exists(lbl_path_1) else lbl_path_2

            if os.path.exists(lbl_path):
                with open(lbl_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                    if len(self.valid_data) == 0:
                        print("--- [첫 번째 데이터 JSON 구조 샘플] ---")
                        print(json.dumps(data, indent=2, ensure_ascii=False)[:300] + "...\n")

                    disease_code = data.get('annotations', {}).get('disease', 0)
                    label = 0 if str(disease_code) == '0' or disease_code == 0 else 1
                    self.valid_data.append((img_name, label))

        print(f"로드 완료: 총 {len(self.valid_data)}개의 데이터 쌍을 확인했습니다.")

    def __len__(self):
        return len(self.valid_data)

    def __getitem__(self, idx):
        img_name, label = self.valid_data[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

try:
    target_dataset = AIHubMatchedDataset(root_dir=TARGET_DATA_DIR, transform=target_transform)
    target_loader = DataLoader(target_dataset, batch_size=16, shuffle=True)

    sample_images, sample_labels = next(iter(target_loader))
    print(f"데이터 로더 생성 성공. 이미지 텐서 형태: {sample_images.shape}, 라벨 형태: {sample_labels.shape}")

except Exception as e:
    print(f"에러 발생: {e}")

--- [첫 번째 데이터 JSON 구조 샘플] ---
{
  "description": {
    "image": "V006_79_0_00_01_01_13_0_a05_20201111_0009_S01_1.JPG",
    "date": "2020/11/11",
    "worker": "",
    "height": 3024,
    "width": 4032,
    "task": 79,
    "type": 0,
    "region": null
  },
  "annotations": {
    "disease": 0,
    "crop": 1,
    "area": 1,
    "g...

로드 완료: 총 100개의 데이터 쌍을 확인했습니다.
데이터 로더 생성 성공. 이미지 텐서 형태: torch.Size([16, 3, 224, 224]), 라벨 형태: torch.Size([16])


In [5]:
# 1. DANN 모델 초기화 (기존 베이스라인 모델 결합)
# num_classes는 기존 PlantVillage의 클래스 개수인 38로 설정합니다.
dann_model = AgriAX_DANN(base_model=model, num_classes=38).to(device)
dann_model.eval()

print("DANN 모델 초기화 완료. 포워드 패스 테스트를 시작합니다.")

# 2. 샘플 데이터를 GPU(또는 CPU) 디바이스로 이동
sample_images = sample_images.to(device)

# 3. 포워드 패스 실행
with torch.no_grad():
    # alpha 값은 Gradient Reversal Layer에 적용되는 가중치입니다.
    class_outputs, domain_outputs = dann_model(sample_images, alpha=1.0)

print("--- 포워드 패스 검증 결과 ---")
print(f"입력 이미지 텐서: {sample_images.shape}")
print(f"클래스 예측 출력 (병충해 분류): {class_outputs.shape}")
print(f"도메인 예측 출력 (실험실 vs 노지): {domain_outputs.shape}")

# 4. 도메인 분류기의 예측 확률 확인
domain_probabilities = torch.softmax(domain_outputs, dim=1)
print(f"\n첫 번째 샘플 이미지의 도메인 예측 확률 (Source vs Target):")
print(domain_probabilities[0].cpu().numpy())

DANN 모델 초기화 완료. 포워드 패스 테스트를 시작합니다.
--- 포워드 패스 검증 결과 ---
입력 이미지 텐서: torch.Size([16, 3, 224, 224])
클래스 예측 출력 (병충해 분류): torch.Size([16, 38])
도메인 예측 출력 (실험실 vs 노지): torch.Size([16, 2])

첫 번째 샘플 이미지의 도메인 예측 확률 (Source vs Target):
[0.4720929 0.5279071]


In [6]:
import torch.optim as optim
import numpy as np

# 1. 손실 함수 및 옵티마이저 설정
# 질병 분류용 손실 함수와 도메인 분류용 손실 함수를 각각 정의합니다.
class_criterion = torch.nn.CrossEntropyLoss()
domain_criterion = torch.nn.CrossEntropyLoss()

# 모델의 전체 파라미터를 업데이트 대상으로 설정합니다.
optimizer = optim.Adam(dann_model.parameters(), lr=1e-4)

# 2. DANN 학습 에폭(Epoch) 함수 정의
def train_dann_epoch(model, source_loader, target_loader, optimizer, device):
    model.train()

    # 미니 데이터셋 환경을 고려하여 더 짧은 로더의 길이에 맞춥니다.
    len_dataloader = min(len(source_loader), len(target_loader))
    source_iter = iter(source_loader)
    target_iter = iter(target_loader)

    running_class_loss = 0.0
    running_domain_loss = 0.0

    for i in range(len_dataloader):
        # Source 데이터 (PlantVillage - 라벨 존재)
        source_data, source_labels = next(source_iter)
        source_data, source_labels = source_data.to(device), source_labels.to(device)

        # Target 데이터 (AI-Hub - 라벨 없이 이미지만 도메인 적응에 사용)
        target_data, _ = next(target_iter)
        target_data = target_data.to(device)

        # 도메인 정답 라벨 생성 (Source 도메인: 0, Target 도메인: 1)
        domain_source_labels = torch.zeros(source_data.size(0), dtype=torch.long).to(device)
        domain_target_labels = torch.ones(target_data.size(0), dtype=torch.long).to(device)

        # GRL(Gradient Reversal Layer)에 적용할 가중치 alpha 계산
        # 학습 초반에는 분류기에 집중하고, 후반부로 갈수록 도메인 적응의 비중을 높입니다.
        p = float(i) / len_dataloader
        alpha = 2. / (1. + np.exp(-10 * p)) - 1

        optimizer.zero_grad()

        # [Step 1] Source 데이터 포워드 패스 (분류 손실 + 도메인 손실 계산)
        class_preds, domain_preds_src = model(source_data, alpha)
        loss_class = class_criterion(class_preds, source_labels)
        loss_domain_src = domain_criterion(domain_preds_src, domain_source_labels)

        # [Step 2] Target 데이터 포워드 패스 (도메인 손실만 계산)
        _, domain_preds_tgt = model(target_data, alpha)
        loss_domain_tgt = domain_criterion(domain_preds_tgt, domain_target_labels)

        # [Step 3] 총 손실 계산 및 역전파
        loss_domain = loss_domain_src + loss_domain_tgt
        total_loss = loss_class + loss_domain

        total_loss.backward()
        optimizer.step()

        running_class_loss += loss_class.item()
        running_domain_loss += loss_domain.item()

    avg_class_loss = running_class_loss / len_dataloader
    avg_domain_loss = running_domain_loss / len_dataloader

    return avg_class_loss, avg_domain_loss

print("DANN 학습 루프 함수 정의가 완료되었습니다.")

DANN 학습 루프 함수 정의가 완료되었습니다.


In [10]:
import os

# kaggle.json 파일 경로 설정 (본인의 환경에 맞게 수정 필요)
KAGGLE_JSON_PATH = '/content/drive/Othercomputers/내 노트북/AgriAX/kaggle.json'

if os.path.exists(KAGGLE_JSON_PATH):
    # Kaggle API 인증 설정
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp "{KAGGLE_JSON_PATH}" /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

    print("Kaggle 데이터 다운로드 및 압축 해제를 시작합니다.")

    # 일반적인 PlantVillage 데이터셋 다운로드 명령어
    # 1일 차에 사용한 특정 데이터셋 주소가 있다면 '-d' 뒷부분을 수정하십시오.
    !kaggle datasets download -d emmarex/plantdisease -p /content/plantvillage --unzip

    # 다운로드된 데이터의 하위 경로 탐색
    base_extract_path = '/content/plantvillage'
    for root, dirs, files in os.walk(base_extract_path):
        if 'PlantVillage' in dirs or 'plantvillage' in dirs:
            target_dir = [d for d in dirs if d.lower() == 'plantvillage'][0]
            SOURCE_DATA_DIR = os.path.join(root, target_dir)
            print(f"데이터 준비 완료. SOURCE_DATA_DIR 경로: {SOURCE_DATA_DIR}")
            break
else:
    print("kaggle.json 파일을 찾을 수 없습니다. 구글 드라이브 내 위치를 확인하십시오.")

Kaggle 데이터 다운로드 및 압축 해제를 시작합니다.
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:06<00:00, 115MB/s]

데이터 준비 완료. SOURCE_DATA_DIR 경로: /content/plantvillage/PlantVillage


In [11]:
import os
from torchvision import datasets
from torch.utils.data import DataLoader

# 1. Source 데이터 로더(PlantVillage) 생성
SOURCE_DATA_DIR = '/content/plantvillage/PlantVillage'

try:
    source_dataset = datasets.ImageFolder(root=SOURCE_DATA_DIR, transform=target_transform)
    source_loader = DataLoader(source_dataset, batch_size=16, shuffle=True)
    print(f"Source 데이터 로더 생성 완료. 총 {len(source_dataset)}개의 이미지를 로드했습니다.")
except Exception as e:
    print(f"Source 데이터 로드 실패: {e}")

# 2. DANN 시범 학습 실행
EPOCHS = 2

if 'source_loader' in locals() and 'target_loader' in locals():
    print("DANN 시범 학습을 시작합니다. (2 Epochs)")
    for epoch in range(EPOCHS):
        c_loss, d_loss = train_dann_epoch(dann_model, source_loader, target_loader, optimizer, device)
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Class Loss: {c_loss:.4f}, Domain Loss: {d_loss:.4f}")
    print("시범 학습 루프가 정상적으로 종료되었습니다.")
else:
    print("데이터 로더가 준비되지 않았습니다. source_loader와 target_loader 변수를 확인하십시오.")

Source 데이터 로더 생성 완료. 총 20638개의 이미지를 로드했습니다.
DANN 시범 학습을 시작합니다. (2 Epochs)
Epoch [1/2] - Class Loss: 3.0786, Domain Loss: 1.3314
Epoch [2/2] - Class Loss: 1.7928, Domain Loss: 1.2344
시범 학습 루프가 정상적으로 종료되었습니다.
